# Entrenamiento independiente de B4

B4 es el modelo principal de la escalera: potenciación del gradiente por histogramas (`HistGradientBoostingClassifier`).
Este cuaderno lo entrena para los escenarios A, B y C en dos partes:

1. **Búsqueda única de hiperparámetros** con Optuna y validación cruzada agrupada por partida.
2. **Reentrenamiento final** de las finalistas con todo el `train` de cada escenario, comparadas en su `val`.

`test` no se toca aquí: se usa una sola vez en `05FinalModelChecks`.

**Requisitos:** servidor de Colab **CPU + High-RAM** (8 núcleos, ~51 GB), sesión de Google para montar Drive y, si los datos no están en `/content/datos`, el token de Kaggle propio.

**Resultados** en `MyDrive/Applied-AI-Agent-Project/model_checkpoints/b4_independiente/busqueda_v3/`:

| Carpeta | Contenido |
|---|---|
| `busqueda/` | Estudio de Optuna, cada trial y pliegue (`b4_cv_trials.csv`) y finalistas (`b4_candidatas.csv`) |
| `final_4000/escenario_X/` | **Modelos finales de B4** (`.joblib`) y sus métricas en `val` |
| `escenario_X/` | Primera corrida con tope de 2000 árboles; se conserva solo como registro |

Si Colab se interrumpe, vuelve a ejecutar las celdas en orden: todo se reanuda desde el último punto guardado.

In [ ]:
from pathlib import Path
import sys
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
CHECKPOINT_ROOT = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints/b4_independiente')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo pull origin 04MainAIModelTrainingAndDevelopment
%cd /content/repo
!pip install -q -r requirements.txt
if '/content/repo/src' not in sys.path:
    sys.path.insert(0, '/content/repo/src')
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
from sklearn import __version__ as sklearn_version
assert tuple(map(int, sklearn_version.split('.')[:2])) >= (1, 7), 'B4 necesita scikit-learn >=1.7; reinicia el runtime.'
DATA_DIR = Path('/content/datos')
from ptcg_wp.ingest import load_days
dias = load_days('configs/dias.yaml')
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(ruta.exists() for ruta in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(ruta.exists() for ruta in esperados), 'Faltan tablas: revisa Kaggle y vuelve a ejecutar setup.'
print('Entorno B4 listo | sklearn:', sklearn_version, '| checkpoints:', CHECKPOINT_ROOT)

In [ ]:
# DIAGNOSTICO DE LA MAQUINA
# PROPOSITO: confirmar que el servidor tiene los nucleos y la RAM esperados antes de entrenar B4.
# HistGradientBoosting usa todos los nucleos: con 2 nucleos cada ajuste tarda unas 4 veces mas.
# ESPERADO: 8 nucleos y ~51 GiB de RAM (CPU + High-RAM). Si sale 2 nucleos y ~12.7 GiB,
# el servidor NO es High-RAM: crear otro servidor antes de seguir.
import os
import psutil
ram_gib = psutil.virtual_memory().total / 1024**3
print("nucleos de CPU:", os.cpu_count(), "| RAM total (GiB):", round(ram_gib, 1))
assert os.cpu_count() >= 8, "Servidor con pocos nucleos: crear uno CPU + High-RAM"

## Parte 1: búsqueda de hiperparámetros

- **Una sola búsqueda** (Optuna, TPE) para los tres escenarios, con validación cruzada de 3 pliegues agrupada por partida.
- Solo usa las **11 285 partidas que son `train` en A, B y C a la vez**: los hiperparámetros nunca ven partidas de `val`, `calib` o `test` de ningún escenario.
- Un solo estudio optimiza log loss (regla de puntuación propia). AUC y Brier se registran en cada trial, y las finalistas por cada métrica salen del mismo estudio.
- 20 trials, con tope de 2000 árboles y parada temprana (paciencia 30) sobre un grupo de vigilancia separado. Con este tamaño de búsqueda, la parada temprana se activa casi siempre antes del tope.
- Sin límite de tiempo: si Colab se corta, se reanuda desde el último pliegue guardado en Drive.
- Duración medida con 8 núcleos: unas 2 h 20 min. Si `b4_candidatas.csv` ya existe, la celda se salta la búsqueda.

**Resultado:** el trial 20 fue el mejor en las tres métricas (AUC 0.838, log loss 0.488 y Brier 0.163 en validación cruzada), así que hay una sola finalista.

In [ ]:
# B4 PARTE 1 - BUSQUEDA UNICA DE HIPERPARAMETROS
# PROPOSITO: buscar hiperparametros UNA sola vez (Optuna TPE, validacion cruzada de 3 pliegues
# agrupada por partida) usando SOLO partidas que son train en A, B y C a la vez. Asi ningun
# hiperparametro vio partidas de val, calib o test de ningun escenario.
# - Un solo estudio optimiza log loss; AUC y Brier de cada trial quedan registrados y definen
#   las finalistas sin ajustes extra.
# - Tope de 2000 arboles con parada temprana sobre un grupo de vigilancia separado.
# - Todo se guarda en Drive por pliegue: si Colab se corta, volver a ejecutar reanuda.
# - NO cambiar estos parametros: estan fijados en el manifiesto del estudio guardado en Drive.
# ESPERADO: 11 285 partidas de busqueda. Si b4_candidatas.csv ya existe, la busqueda se omite
# y solo se imprimen las finalistas (trial 20: AUC 0.838, log loss 0.488, Brier 0.163 en CV).
from tempfile import TemporaryDirectory
import gc
import faulthandler
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.b4 import materialize_split, b4_optuna_cv
from ptcg_wp.checkpoints import atomic_csv
from ptcg_wp.features import feature_columns

features = feature_columns(pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names)
B4_TRIALS = 20
B4_SPLITS = 3
B4_SEARCH_FRACTION = 1.0
B4_MAX_ITER = 2000
B4_PATIENCE = 30
B4_OBJECTIVE = 'log_loss'
MAX_RAM_GIB = 10.0
SEED = 2026
RUN_ROOT = CHECKPOINT_ROOT / 'busqueda_v3'
CANDIDATES_PATH = RUN_ROOT / 'busqueda' / 'b4_candidatas.csv'

# Partidas que son train en los tres escenarios: las unicas que ve la busqueda
part = pd.read_parquet(DATA_DIR / 'particiones.parquet')
comunes = part.loc[(part['split_A'] == 'train') & (part['split_B'] == 'train')
                   & (part['split_C'] == 'train'), 'game_id']
print('caracteristicas:', len(features), '| partidas de busqueda (train en A, B y C):', len(comunes))

if not CANDIDATES_PATH.exists():
    trace_file = open('/content/b4_pipeline_tracebacks.log', 'a', buffering=1)
    faulthandler.enable(file=trace_file)
    faulthandler.dump_traceback_later(600, repeat=True, file=trace_file)
    try:
        print('B4: busqueda de hiperparametros', flush=True)
        with TemporaryDirectory(prefix='b4-busqueda-', dir='/content') as cache:
            stream = ScenarioStream(DATA_DIR, 'A', tuple(features), batch_size=32768)
            train = materialize_split(stream, features, 'train', cache)
            try:
                codigos = np.flatnonzero(train.games.isin(comunes))
                _, candidates = b4_optuna_cv(
                    train, n_splits=B4_SPLITS, n_trials=B4_TRIALS, timeout=None, seed=SEED,
                    max_iter=B4_MAX_ITER, patience=B4_PATIENCE, max_ram_gib=MAX_RAM_GIB,
                    output_dir=RUN_ROOT / 'busqueda',
                    study_cache_dir='/content/ptcg_optuna/b4_busqueda_v3',
                    verbose=True, objective_metric=B4_OBJECTIVE,
                    search_fraction=B4_SEARCH_FRACTION, search_groups=codigos,
                )
                atomic_csv(candidates.to_dict('records'), CANDIDATES_PATH)
            finally:
                train.close()
            del stream
        gc.collect()
    finally:
        faulthandler.cancel_dump_traceback_later()
        faulthandler.disable()
        trace_file.close()

# Se leen siempre del CSV para que todas las sesiones usen exactamente las mismas finalistas
candidates = pd.read_csv(CANDIDATES_PATH)
print(candidates.to_string(index=False), flush=True)

## Parte 2: reentrenamiento final

Cada escenario reentrena las finalistas con **todo** su `train` y las mide en su `val`:

1. Un ajuste con parada temprana sobre el 15 % de las partidas de `train` decide el número de árboles.
2. Un segundo ajuste con todo `train` usa ese número de árboles.

**Tope de 4000 árboles.** En una primera corrida con tope 2000 (carpetas `escenario_X`), los tres escenarios llegaron al tope: el modelo seguía mejorando. Con tope 4000, la parada temprana convergió sola (2888, 2370 y 2714 árboles) y las métricas en `val` cambiaron menos de 0.002. Los modelos de `final_4000` son los **modelos finales de B4** para los tres escenarios, con una sola regla para todos.

Duración medida con 8 núcleos: unas 2 h 40 min. Si un escenario ya terminó, se omite.

In [ ]:
# B4 PARTE 2 - REENTRENAMIENTO FINAL CON TODO EL TRAIN DE CADA ESCENARIO (TOPE 4000)
# PROPOSITO: reentrenar la finalista de la busqueda con TODO el train de cada escenario y
# medirla en su val. La parada temprana (15 % de partidas de train como vigilancia) decide el
# numero de arboles, con un tope de 4000 que no se alcanza (convergio entre 2370 y 2888).
# - Requiere haber ejecutado la celda de la parte 1 (define features, candidates y parametros).
# - Si un escenario ya termino, se omite (reanuda desde lo guardado en Drive).
# - test NO se toca.
# ESPERADO (medido): A arboles=2888 AUC=0.8346 | B arboles=2370 AUC=0.7863 |
# C arboles=2714 AUC=0.8355. Modelos en busqueda_v3/final_4000/escenario_X/*.joblib.
from ptcg_wp.b4 import fit_b4_candidates_val

B4_MAX_ITER_FINAL = 4000
FINAL_ROOT = RUN_ROOT / 'final_4000'

for scenario in ('A', 'B', 'C'):
    output_dir = FINAL_ROOT / f'escenario_{scenario}'
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f'B4 {scenario}: preparando train y val', flush=True)
    with TemporaryDirectory(prefix=f'b4-final-{scenario}-', dir='/content') as cache:
        stream = ScenarioStream(DATA_DIR, scenario, tuple(features), batch_size=32768)
        train = materialize_split(stream, features, 'train', cache)
        val = materialize_split(stream, features, 'val', cache)
        try:
            val_metrics = fit_b4_candidates_val(
                train, val, candidates, seed=SEED, max_iter=B4_MAX_ITER_FINAL,
                patience=B4_PATIENCE, max_ram_gib=MAX_RAM_GIB,
                output_dir=output_dir, scenario=scenario, verbose=True,
            )
            atomic_csv(val_metrics.to_dict('records'), output_dir / 'b4_val_finalistas.csv')
            g = val_metrics[val_metrics['etapa'] == 'global'].iloc[0]
            print(f"B4 {scenario}: terminado | arboles={g['n_iter_vigilancia']} | "
                  f"AUC={g['auc']:.4f} log_loss={g['log_loss']:.4f} brier={g['brier']:.4f}", flush=True)
        finally:
            train.close()
            val.close()
        del stream
    gc.collect()
print('B4 listo. Modelos finales en:', FINAL_ROOT)

In [ ]:
# B4 - RESUMEN DE RESULTADOS EN VAL
# PROPOSITO: mostrar las metricas de los modelos finales de B4 (final_4000) en val, globales y
# por etapa, y compararlas con la primera corrida de tope 2000 (registro de estabilidad).
# n_iter_vigilancia = arboles elegidos por la parada temprana.
# ESPERADO (medido, global): A AUC 0.835 | B AUC 0.786 | C AUC 0.836; la diferencia con la
# corrida de tope 2000 es menor a 0.002 en todas las metricas.
columnas = ['escenario', 'etapa', 'auc', 'log_loss', 'brier', 'n_partidas', 'n_estados', 'n_iter_vigilancia']
comparacion = []
for scenario in ('A', 'B', 'C'):
    final = pd.read_csv(FINAL_ROOT / f'escenario_{scenario}' / 'b4_val_finalistas.csv')
    print(final[columnas].round(4).to_string(index=False))
    print('modelo final:', [p.name for p in (FINAL_ROOT / f'escenario_{scenario}').glob('*.joblib')], '\n')
    previa_path = RUN_ROOT / f'escenario_{scenario}' / 'b4_val_finalistas.csv'
    if previa_path.exists():
        previa = pd.read_csv(previa_path)
        a = final[final['etapa'] == 'global'].iloc[0]
        b = previa[previa['etapa'] == 'global'].iloc[0]
        comparacion.append({'escenario': scenario,
                            'arboles_2000': int(b['n_iter_vigilancia']), 'arboles_4000': int(a['n_iter_vigilancia']),
                            **{f'dif_{m}': round(a[m] - b[m], 4) for m in ('auc', 'log_loss', 'brier')}})
if comparacion:
    print('Tope 4000 menos tope 2000 (global, val):')
    print(pd.DataFrame(comparacion).to_string(index=False))

## Resultados versionados en el repositorio

Al terminar el entrenamiento, los resultados y los modelos finales se **copiaron manualmente** desde
Drive a `results/baseline4/` del repositorio, como referencia permanente para las siguientes etapas
(evaluación en `test`, calibración, SHAP y agente de la Etapa 3). Este cuaderno no hace la copia:
se hizo a mano y se subió con un commit desde la PC.

| Desde Drive (`busqueda_v3/`) | Hacia el repo (`results/baseline4/`) |
|---|---|
| `busqueda/b4_candidatas.csv`, `b4_cv_trials.csv`, `b4_optuna.sqlite3` | `busqueda/` |
| `final_4000/escenario_X/b4_auc_CVtrial20.joblib`, `b4_val_finalistas.csv` y `b4_val_manifest.json` | `final_4000/escenario_X/` |
| `escenario_X/b4_val_finalistas.csv` y `b4_val_manifest.json` (tope 2000) | `registro_tope_2000/escenario_X/` |

Los modelos `.joblib` (unos 10 MB cada uno) se versionan gracias a la excepción
`!results/**/*.joblib` del `.gitignore`. Los detalles, las métricas y el código para cargarlos están
en `results/baseline4/README.md`.

Si se vuelve a entrenar B4, hay que repetir esta copia para que el repositorio refleje los nuevos resultados.